# Pooled Analysis: HBN + Adult Rest/Movie + Infant Rest/Movie (with bootstrapping)

> **Reproducibility note (read before running):** this notebook is one of a set of
> `0*_*.ipynb` notebooks split out of the original `compiled_results_analysis.ipynb`
> for the publication reproducibility bundle, meant to be run with the repo root (`task_dim/`) as
> the working directory. Each notebook is self-contained (re-loads its own
> inputs rather than relying on variables from other notebooks). A global `SEED = 4` is set at
> the top of the setup cell so every resampling/permutation step below is deterministic.
>
> **Inputs used here are pre-computed.** Where a cell loads a CSV/NIfTI file, a comment states
> which upstream script produced it. A few inputs this notebook depends on
> (`compiled/info/combined_participant_info.csv`, `compiled/results/*_compiled_results.csv`,
> `compiled/results/combined_corrs.csv`, `compiled/results/combined_corr_analyses.csv`) 

> **Fixes applied vs. the original notebook** (this section had the most bugs/dead code of the
> whole `compiled_results_analysis.ipynb`):
> 1. The cell building `res` (per-subject within-subject Spearman rho between ISC and delta-ID)
>    was commented out in the original, re-loading a cached CSV instead. Re-enabled here using
>    `stats_helpers.within_subject_spearman` (still present in `stats_helpers.py`), with
>    `random_state=SEED` added (the original call had no seed).
> 2. The bootstrap-distribution plot referenced undefined variables `ci_lo`, `ci_hi`, and
>    `rho_obs` -- these were typos for `rho_ci_lower`, `rho_ci_upper`, and `rho_observed` (the
>    actual return values of `bootstrap_rho_isc_deltaID_unmatched`). Fixed here.
> 3. `bootstrap_rho_isc_deltaID_unmatched(..., seed=4)` is now called with `seed=SEED` for
>    consistency with the rest of this reproducibility bundle.
> 4. The mixed-effects model cells that used an undefined `new_data` (`model_log.predict(new_data)`)
>    and undefined `ols_model` (heteroscedasticity / robust-SE diagnostics) were dead code in the
>    original -- they would raise `NameError` if run and nothing downstream used their outputs.
>    They are dropped here; the one place a prediction at the mean infant age is actually needed
>    (the final trajectory plot) computes it inline from `model_log` directly.
> 5. The final trajectory plot's bootstrap confidence band (`boot_preds`) was computed by a loop
>    that was commented out, then immediately used anyway (`np.nanpercentile(boot_preds, ...)`) --
>    this would have raised `NameError`. The loop is re-enabled here (`n_boot=1000`, seeded per
>    iteration via `random_state=i`, so it is fully deterministic).
> 6. A near-duplicate 8-quantile-bin diagnostic bar plot is dropped in favor of the actual
>    downstream binning (`make_groups`) that `summary_df` is built from.

> **Depends on:** `HBN/parcelwise_difference_ISC_IDE.csv`, built in `05_hbn.ipynb`. Run that notebook first (or ensure the CSV is already present).

In [1]:
import numpy as np
import pandas as pd
import os, sys, glob
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
import scipy
import scipy.stats as stats
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.stats import ttest_rel, ttest_ind, wilcoxon, spearmanr
import nilearn
from nilearn import plotting, image, datasets
from nilearn.maskers import NiftiMasker
from nilearn.mass_univariate import permuted_ols
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.anova import anova_lm
import matplotlib.gridspec as gridspec
from patsy import dmatrix
from sklearn.utils import resample
import pingouin as pg
from matplotlib.patches import Patch
from matplotlib import rcParams
import matplotlib.ticker as mticker
from scipy.stats import gaussian_kde

# Repo-local helper modules (see CLAUDE.md for the dataset-abstraction pattern)
import plotting_helpers as helper          # surface plots, colormaps, dataset colors
import stats_helpers as stats_helpers      # permutation tests, (parcelwise/LME) regression, FDR correction
import parcelwise_regressions as pwr       # run_regression_analyses, join_results_participant_info, clean_difference_dataframe
import hbn_config as hc                    # hc.AGE_BINS / hc.HBN_AGE_GROUPS used for HBN age binning

rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42

PLOT_DIR = 'compiled/plots'
os.makedirs(PLOT_DIR, exist_ok=True)
ISC_CMAP = 'OrRd'
ID_CMAP = helper.custom_blues_cmap()
dataset_colors = helper.dataset_colors('all')

# Global seed: makes every unseeded resample/permutation call below reproducible.
SEED = 4
np.random.seed(SEED)

In [2]:
# Common helper functions -- now defined once in plotting_helpers.py / stats_helpers.py
# (previously redefined identically in every final_results notebook; see REPRODUCIBILITY.md).
from plotting_helpers import (
    expand_parcellation_to_volume,  # parcel-array -> 3D volume
    load_atlas, get_region_order,   # Schaefer-400 atlas + region label order
    get_average_results, reorder_region_names,  # average/reindex per-region scores
)
from stats_helpers import lme_fit_summary  # MixedLM fit diagnostics (R^2, ICC, AIC/BIC)

## Combine parcelwise delta_ID (rest − movie IDE) and ISC across HBN + Adult Rest/Movie

In [3]:
# INPUT: HBN/parcelwise_difference_ISC_IDE.csv -- built in 05_hbn.ipynb
# INPUT: compiled/results/adult_restmovie_compiled_results.csv (see REPRODUCIBILITY.md)
hbn_difference_df = pd.read_csv('HBN/parcelwise_difference_ISC_IDE.csv')

# === Adult Rest/Movie ===
arm_csv = pd.read_csv('compiled/results/adult_restmovie_compiled_results.csv')
arm_ide = arm_csv[arm_csv['measure'] == 'TPHATE_DiffOp_IDE'].copy()
arm_ide_pivot = arm_ide.pivot_table(index=['subject_id', 'region_name'], columns='task', values='score').reset_index()
arm_ide_pivot = arm_ide_pivot.dropna(subset=['aeronaut', 'rest'])
arm_ide_pivot['delta_ID'] = arm_ide_pivot['rest'] - arm_ide_pivot['aeronaut']

arm_isc = (arm_csv[(arm_csv['measure'] == 'ISC') & (arm_csv['task'] == 'aeronaut')]
           [['subject_id', 'region_name', 'score']].rename(columns={'score': 'ISC'}))
arm_pivot = arm_ide_pivot.merge(arm_isc, on=['subject_id', 'region_name'], how='inner')

arm_age = arm_csv[['subject_id', '_age_raw']].drop_duplicates()
arm_pivot = arm_pivot.merge(arm_age, on='subject_id', how='left').rename(columns={'_age_raw': 'age'})
arm_pivot['dataset'] = 'adult_restmovie'
arm_pivot.to_csv('adult_restmovie/parcelwise_difference_ISC_IDE.csv', index=False)

# === HBN ===
hbn_pivot = hbn_difference_df[['subject_id', 'region_name', 'RestMovieDiff', 'ISC', 'Age']].copy()
hbn_pivot = hbn_pivot.rename(columns={'RestMovieDiff': 'delta_ID', 'Age': 'age'})
hbn_pivot['dataset'] = 'HBN'

# === Combine ===
keep_cols = ['subject_id', 'region_name', 'delta_ID', 'ISC', 'age', 'dataset']
combined_delta_isc = pd.concat([arm_pivot[keep_cols], hbn_pivot[keep_cols]], ignore_index=True)
combined_delta_isc.to_csv('compiled/results/combined_delta_id_isc.csv', index=False)
print(f"Combined: {combined_delta_isc.shape[0]} rows across {combined_delta_isc['dataset'].nunique()} datasets")
print(combined_delta_isc.groupby('dataset')[['delta_ID', 'ISC']].describe())
combined_delta_isc.head()

Combined: 215600 rows across 2 datasets
                 delta_ID                                                 \
                    count      mean       std   min  25%  50%  75%   max   
dataset                                                                    
HBN              211200.0  2.736269  3.763973 -19.0  0.0  2.0  4.0  41.0   
adult_restmovie    4400.0  1.922955  2.833521  -8.0  0.0  2.0  4.0  14.0   

                      ISC                                                    \
                    count      mean       std       min       25%       50%   
dataset                                                                       
HBN              211200.0  0.117065  0.095130 -0.260689  0.047104  0.098501   
adult_restmovie    4400.0  0.155818  0.161189 -0.371718  0.043192  0.125821   

                                     
                      75%       max  
dataset                              
HBN              0.172024  0.580717  
adult_restmovie  0.238477  0.75

,subject_id,region_name,delta_ID,ISC,age,dataset
0,rest_movie_01,17Networks_LH_ContA_Cingm_1,3.0,0.212337,19.0,adult_restmovie
1,rest_movie_01,17Networks_LH_ContA_IPS_1,4.0,0.065532,19.0,adult_restmovie
2,rest_movie_01,17Networks_LH_ContA_IPS_2,9.0,0.191246,19.0,adult_restmovie
3,rest_movie_01,17Networks_LH_ContA_IPS_3,8.0,0.054974,19.0,adult_restmovie
4,rest_movie_01,17Networks_LH_ContA_IPS_4,10.0,0.194707,19.0,adult_restmovie


In [4]:
combined_delta_isc = pd.read_csv('compiled/results/combined_delta_id_isc.csv')
combined_delta_isc['age_months'] = combined_delta_isc['age'] * 12
combined_delta_isc['age_months_z'] = stats.zscore(combined_delta_isc['age_months'])
# z-score delta_ID and ISC within each subject (across parcels) to put them on the same scale
combined_delta_isc['delta_ID_z'] = combined_delta_isc.groupby('subject_id')['delta_ID'].transform(lambda x: stats.zscore(x, nan_policy='omit'))
combined_delta_isc['ISC_z'] = combined_delta_isc.groupby('subject_id')['ISC'].transform(lambda x: stats.zscore(x, nan_policy='omit'))

## Subject-wise ISC vs delta-ID Spearman correlation (HBN + Adult Rest/Movie)

In [6]:
# INPUT: compiled/info/combined_participant_info.csv -- NOTE: removed from this
# repo (participant privacy); see REPRODUCIBILITY.md to supply your own copy.
# Computed with stats_helpers.within_subject_spearman (permutation-based per-subject Spearman
# correlation across parcels; see stats_helpers.permute_pattern for the underlying test).
if os.path.exists('compiled/results/isc_deltaid_subjectwise_correlation.csv'):
    res = pd.read_csv('compiled/results/isc_deltaid_subjectwise_correlation.csv')
else:
    res = stats_helpers.within_subject_spearman(
        combined_delta_isc, 'ISC_z', 'delta_ID_z', subject_col='subject_id', random_state=SEED)
    res = res[['subject_id', 'rho', 'pval', 'zscore']]

    participant_df = pd.read_csv('compiled/info/combined_participant_info.csv')
    participant_df = participant_df[participant_df['dataset'].isin(['HBN', 'AdultRestMovie'])]
    participant_df = participant_df.rename(columns={'participant_id': 'subject_id'})
    res = res.merge(participant_df[['subject_id', 'age_months', 'dataset']], on='subject_id', how='left')
    res['age'] = res['age_months'] / 12
    res.to_csv('compiled/results/isc_deltaid_subjectwise_correlation.csv', index=False)
res.head()

,subject_id,rho,pval,zscore,age_months,dataset,age,age_group
0,rest_movie_01,0.064058,0.208791,1.321249,228.0,AdultRestMovie,19.0,O17
1,rest_movie_02,-0.180248,0.000999,-3.712178,276.0,AdultRestMovie,23.0,O17
2,rest_movie_03,0.256089,0.000999,5.125004,228.0,AdultRestMovie,19.0,O17
3,rest_movie_04,0.075164,0.146853,1.513999,336.0,AdultRestMovie,28.0,O17
4,rest_movie_05,-0.207367,0.000999,-4.119266,348.0,AdultRestMovie,29.0,O17


## Infant group bootstrapping

Two independent pools of infants (sleep, movie watch). Generate a distribution of group-level
delta-ID maps that reflect the uncertainty of unmatched groups, to get a single
rho(ISC, delta-ID) with a confidence interval that exists on the same axis as the subject-level
values from HBN / Adults. Bootstrapping resamples group means rather than individual subjects:
each iteration resamples sleepers & computes the mean ID map, resamples movie-watchers & computes
the mean ID map, subtracts, and correlates with the group ISC map.

In [7]:
# INPUT: compiled/results/infant_restmovie_compiled_results.csv (see REPRODUCIBILITY.md)
inf_df = pd.read_csv('compiled/results/infant_restmovie_compiled_results.csv', index_col=0)
# NOTE: compiled/info/combined_participant_info.csv is removed from this repo (participant privacy); see REPRODUCIBILITY.md to supply your own copy.
inf_ages = pd.read_csv('compiled/info/combined_participant_info.csv')
inf_ages = inf_ages[inf_ages['dataset'] == 'InfantRestMovie']['age_months'].values
inf_df.head()

,measure,region_name,score,task,subject_id,dataset,age_months
Unnamed: 0,,,,,,,
83200,TPHATE_DiffOp_IDE,17Networks_LH_VisCent_ExStr_1,4.0,aeronaut,s8687_1_3,infant_restmovie,11.0
83201,TPHATE_DiffOp_IDE,17Networks_LH_VisCent_ExStr_2,7.0,aeronaut,s8687_1_3,infant_restmovie,11.0
83202,TPHATE_DiffOp_IDE,17Networks_LH_VisCent_ExStr_3,6.0,aeronaut,s8687_1_3,infant_restmovie,11.0
83203,TPHATE_DiffOp_IDE,17Networks_LH_VisCent_ExStr_4,7.0,aeronaut,s8687_1_3,infant_restmovie,11.0
83204,TPHATE_DiffOp_IDE,17Networks_LH_VisCent_ExStr_5,5.0,aeronaut,s8687_1_3,infant_restmovie,11.0


In [8]:
def bootstrap_rho_isc_deltaID_unmatched(id_sleep, id_movie, isc_movie, n_iterations=10000, seed=SEED,
                                         zscore_within_iteration=True):
    """
    Parameters
    ----------
    id_sleep  : (n_sleepers, n_parcels)
    id_movie  : (n_movie,    n_parcels)
    isc_movie : (n_parcels,)  -- group-mean ISC, treated as fixed
    n_iterations : int
    zscore_within_iteration : bool
        If True, z-score deltaID and ISC across parcels within each bootstrap iteration before
        computing rho (matches the within-subject z-scoring done for matched participants,
        making the infant rho directly comparable). If False, raw Spearman rho is computed.

    Returns
    -------
    rho_observed, zscore_observed, rho_boot, rho_ci_lower, rho_ci_upper,
    z_observed, z_ci_lower, z_ci_upper, z_boot
    """
    rng_boot = np.random.default_rng(seed)
    n_sleepers, n_parcels = id_sleep.shape
    n_movie = id_movie.shape[0]

    mean_sleep_obs = id_sleep.mean(axis=0)
    mean_movie_obs = id_movie.mean(axis=0)
    delta_obs = mean_sleep_obs - mean_movie_obs

    if zscore_within_iteration:
        delta_obs_z = stats.zscore(delta_obs)
        isc_z = stats.zscore(isc_movie)
    else:
        delta_obs_z = delta_obs
        isc_z = isc_movie

    _, rho_observed, z_observed = stats_helpers.permute_pattern(
        delta_obs_z, isc_z, n_permutations=n_iterations, random_state=seed)

    rho_boot = np.empty(n_iterations)
    z_boot = np.empty(n_iterations)
    for i in range(n_iterations):
        idx_sleep = rng_boot.integers(0, n_sleepers, size=n_sleepers)
        idx_movie = rng_boot.integers(0, n_movie, size=n_movie)

        mean_sleep_b = id_sleep[idx_sleep].mean(axis=0)
        mean_movie_b = id_movie[idx_movie].mean(axis=0)
        delta_b = mean_sleep_b - mean_movie_b

        delta_b_z = stats.zscore(delta_b) if zscore_within_iteration else delta_b
        _, rboot, zboot = stats_helpers.permute_pattern(
            delta_b_z, isc_z, n_permutations=n_iterations, random_state=seed)
        rho_boot[i] = rboot
        z_boot[i] = zboot
        if i % 100 == 0:
            print(f"Bootstrap iteration {i}/{n_iterations} completed")

    rho_ci_lower = np.percentile(rho_boot, 2.5)
    rho_ci_upper = np.percentile(rho_boot, 97.5)
    rho_boot_mean = np.mean(rho_boot); rho_boot_std = np.std(rho_boot)
    zscore_rho_observed = (rho_observed - rho_boot_mean) / rho_boot_std

    z_ci_lower = np.percentile(z_boot, 2.5)
    z_ci_upper = np.percentile(z_boot, 97.5)
    z_boot_mean = np.mean(z_boot); z_boot_std = np.std(z_boot)
    zscore_z_observed = (z_observed - z_boot_mean) / z_boot_std

    print(f'Rho: observed = {rho_observed:.4f} (95% CI: [{rho_ci_lower:.4f}, {rho_ci_upper:.4f}]), '
          f'z-score = {zscore_rho_observed:.4f}')
    print(f'Z: observed = {z_observed:.4f} (95% CI: [{z_ci_lower:.4f}, {z_ci_upper:.4f}]), '
          f'z-score = {zscore_z_observed:.4f}')
    return rho_observed, rho_ci_lower, rho_ci_upper, rho_boot, z_observed, z_ci_lower, z_ci_upper, z_boot

In [9]:
id_sleep = inf_df[(inf_df['task'] == 'sleep') & (inf_df['measure'] == 'TPHATE_DiffOp_IDE')].pivot_table(index='subject_id', columns='region_name', values='score').values
id_movie = inf_df[(inf_df['task'] == 'aeronaut') & (inf_df['measure'] == 'TPHATE_DiffOp_IDE')].pivot_table(index='subject_id', columns='region_name', values='score').values
isc_movie = inf_df[(inf_df['task'] == 'aeronaut') & (inf_df['measure'] == 'ISC')].groupby('region_name')['score'].mean().values

id_sleep = stats.zscore(id_sleep, axis=1, nan_policy='omit')
id_movie = stats.zscore(id_movie, axis=1, nan_policy='omit')
isc_movie = stats.zscore(isc_movie, nan_policy='omit')

print(f"ID_sleep shape: {id_sleep.shape}, ID_movie shape: {id_movie.shape}, ISC_movie shape: {isc_movie.shape}")

rho_observed, rho_ci_lower, rho_ci_upper, rho_boot, z_observed, z_ci_lower, z_ci_upper, z_boot = bootstrap_rho_isc_deltaID_unmatched(
    id_sleep, id_movie, isc_movie, n_iterations=1000, seed=SEED)

p_twotailed_rho = 2 * min(np.mean(rho_boot >= 0), np.mean(rho_boot <= 0))
print(f"  Bootstrap p (two-tailed, H0: rho=0): {p_twotailed_rho:.4f}")
p_twotailed_z = 2 * min(np.mean(z_boot >= 0), np.mean(z_boot <= 0))
print(f"  Bootstrap p (two-tailed, H0: z=0): {p_twotailed_z:.4f}")

ID_sleep shape: (19, 400), ID_movie shape: (26, 400), ISC_movie shape: (400,)
Bootstrap iteration 0/1000 completed
Bootstrap iteration 100/1000 completed
Bootstrap iteration 200/1000 completed
Bootstrap iteration 300/1000 completed
Bootstrap iteration 400/1000 completed
Bootstrap iteration 500/1000 completed
Bootstrap iteration 600/1000 completed
Bootstrap iteration 700/1000 completed
Bootstrap iteration 800/1000 completed
Bootstrap iteration 900/1000 completed
Rho: observed = 0.1480 (95% CI: [-0.0337, 0.2870]), z-score = 0.2973
Z: observed = 2.8251 (95% CI: [-0.6209, 5.5962]), z-score = 0.2596
  Bootstrap p (two-tailed, H0: rho=0): 0.1420
  Bootstrap p (two-tailed, H0: z=0): 0.1320


In [ ]:
# Format group-mean delta_ID/ISC to match combined_delta_isc's columns.
# Infant subjects are unmatched (sleep vs aeronaut are different people), so delta_ID is
# computed from group means rather than within-subject differences. Parcels are in
# alphabetical region_name order (pandas default for pivot/groupby above).
region_names_inf = sorted(inf_df['region_name'].unique())

inf_parcelwise = pd.DataFrame({
    'subject_id': 'infant_group',
    'region_name': region_names_inf,
    'delta_ID': id_sleep.mean(axis=0) - id_movie.mean(axis=0),
    'ISC': isc_movie,
    'age': inf_df['age_months'].mean(),
    'dataset': 'infant',
})

combined_delta_isc_all = pd.concat([combined_delta_isc, inf_parcelwise], ignore_index=True)
combined_delta_isc_all.to_csv('compiled/results/combined_delta_id_isc_all_datasets.csv', index=False)
print(f"Rows by dataset:\n{combined_delta_isc_all.groupby('dataset').size()}")
combined_delta_isc_all.head()

### Panel figure: bootstrap distribution + parcel scatter + developmental trajectory

In [ ]:
mean_infant_age_months = inf_df['age_months'].mean()

infant_row = pd.DataFrame([{
    'subject_id': 'infant_group', 'rho': rho_observed, 'pval': np.nan, 'zscore': z_observed,
    'age_months': mean_infant_age_months, 'dataset': 'infant', 'is_group': True,
    'rho_ci_lo': rho_ci_lower, 'rho_ci_hi': rho_ci_upper,
    'zscore_ci_lo': z_ci_lower, 'zscore_ci_hi': z_ci_upper, 'age_group': 'infant',
}])

res_combined = res.copy()
res_combined['is_group'] = False
res_combined['rho_ci_lo'] = np.nan; res_combined['rho_ci_hi'] = np.nan
res_combined['zscore_ci_lo'] = np.nan; res_combined['zscore_ci_hi'] = np.nan

df_combined = pd.concat([res_combined, infant_row], ignore_index=True)
df_combined.to_csv('compiled/results/isc_deltaid_rho_developmental.csv', index=False)
print(f"Combined: {len(df_combined)} rows ({df_combined[df_combined['is_group']==False].shape[0]} subjects + 1 infant group point)")
print(df_combined.groupby('dataset')[['rho', 'zscore']].describe())

fig = plt.figure(figsize=(14, 5))
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)

# Panel A: bootstrap distribution
ax_a = fig.add_subplot(gs[0])
ax_a.hist(rho_boot, bins=60, color='#4477AA', alpha=0.75, edgecolor='none')
ax_a.axvline(rho_observed, color='black', lw=2, label=f'Observed rho = {rho_observed:.3f}')
ax_a.axvline(rho_ci_lower, color='#EE6677', lw=1.5, ls='--', label=f'95% CI [{rho_ci_lower:.3f}, {rho_ci_upper:.3f}]')
ax_a.axvline(rho_ci_upper, color='#EE6677', lw=1.5, ls='--')
ax_a.axvline(0, color='gray', lw=1, ls=':', label='rho = 0')
ax_a.set_xlabel('Bootstrap rho(ISC, deltaID)', fontsize=11)
ax_a.set_ylabel('Count', fontsize=11)
ax_a.set_title('Infant bootstrap distribution\n(unmatched sleep vs. movie)', fontsize=11)
ax_a.legend(fontsize=8)

# Panel B: group-mean maps scatter (ISC vs deltaID across parcels)
ax_b = fig.add_subplot(gs[1])
delta_obs_full = id_sleep.mean(0) - id_movie.mean(0)
ax_b.scatter(isc_movie, delta_obs_full, alpha=0.5, s=15, c='#4477AA', edgecolors='none')
m, b_lin = np.polyfit(isc_movie, delta_obs_full, 1)
xs = np.linspace(isc_movie.min(), isc_movie.max(), 100)
ax_b.plot(xs, m * xs + b_lin, 'k-', lw=1.5)
ax_b.set_xlabel('ISC (group mean)', fontsize=11)
ax_b.set_ylabel('deltaID sleep-movie (group mean)', fontsize=11)
ax_b.set_title(f'Parcel-wise ISC vs deltaID\nrho = {rho_observed:.3f}', fontsize=11)

# Panel C: developmental trajectory with real per-subject data + infant group
ax_c = fig.add_subplot(gs[2])
dataset_colors_panelc = {'HBN': '#AAAAAA', 'adult_restmovie': '#888888', 'infant': '#4477AA'}
subj_df = df_combined[df_combined['is_group'] == False]
for ds, grp in subj_df.groupby('dataset'):
    ax_c.scatter(grp['age_months'], grp['zscore'], c=dataset_colors_panelc.get(ds, '#CCCCCC'),
                 alpha=0.4, s=10, zorder=1, label=ds)

inf_row = df_combined[df_combined['is_group'] == True].iloc[0]
ax_c.errorbar(inf_row['age_months'], inf_row['zscore'],
    yerr=[[inf_row['zscore'] - inf_row['zscore_ci_lo']], [inf_row['zscore_ci_hi'] - inf_row['zscore']]],
    fmt='o', color='#4477AA', markersize=9, capsize=5, lw=2, zorder=3,
    label=f'Infant group (bootstrap)\nz = {inf_row["zscore"]:.2f} [{inf_row["zscore_ci_lo"]:.2f}, {inf_row["zscore_ci_hi"]:.2f}]')
ax_c.axhline(0, color='gray', lw=1, ls='--', alpha=0.7)
ax_c.set_xlabel('Age (months)', fontsize=11)
ax_c.set_ylabel('z(rho(ISC, deltaID))', fontsize=11)
ax_c.set_title('ISC-deltaID coupling across development', fontsize=11)
ax_c.legend(fontsize=8)

plt.suptitle('Developmental trajectory of ISC-deltaID spatial coupling', fontsize=12, y=1.02)
plt.savefig(f'{PLOT_DIR}/isc_deltaid_developmental_trajectory.pdf', format='pdf', transparent=True, bbox_inches='tight')
plt.show()

### Bin subjects into data-driven age groups + build developmental summary table

In [ ]:
def make_groups(age):
    if age < 8: return 'U08'
    if age < 9.5: return '8-9.5'
    if age < 10.5: return '9.5-10.5'
    if age < 12: return '10.5-12'
    if age < 13.5: return '12-13.5'
    if age < 15: return '13.5-15'
    if age < 17: return '15-17'
    return 'O17'

res['age_group'] = res['age'].apply(make_groups)
order = ['U08', '8-9.5', '9.5-10.5', '10.5-12', '12-13.5', '13.5-15', '15-17', 'O17']
sns.pointplot(x='age_group', y='zscore', data=res, color='lightgray', order=order)
age_group_counts = res['age_group'].value_counts().reindex(order)
age_group_labels = [f'{age_group}\n(n={count})' for age_group, count in age_group_counts.items()]
plt.xticks(ticks=range(len(age_group_labels)), labels=age_group_labels)
plt.show()

In [ ]:
summary_df = pd.DataFrame(columns=['age_group', 'mean_z', 'ci_lower', 'ci_upper', 'sem', 'std', 'count',
                                    'mean_age_months', 'mean_rho', 'rho_ci_lower', 'rho_ci_upper'])
for ag, grp in res.groupby('age_group'):
    mean = grp['zscore'].mean(); count = grp['zscore'].count(); std = grp['zscore'].std()
    age_mean = grp['age_months'].mean()
    sem = std / np.sqrt(count)
    ci_lower = mean - 1.96 * sem; ci_upper = mean + 1.96 * sem
    mean_rho = grp['rho'].mean()
    rho_sem = grp['rho'].std() / np.sqrt(count)
    rho_ci_lower_g = mean_rho - 1.96 * rho_sem; rho_ci_upper_g = mean_rho + 1.96 * rho_sem
    summary_df.loc[len(summary_df)] = [ag, mean, ci_lower, ci_upper, sem, std, count, age_mean,
                                        mean_rho, rho_ci_lower_g, rho_ci_upper_g]

# Add the infant supergroup as its own row (from the main bootstrap above)
n_boot = 1000
rho_ci_lower_boot = np.percentile(rho_boot, 2.5)
rho_ci_upper_boot = np.percentile(rho_boot, 97.5)
summary_df.loc[len(summary_df)] = ['infant', z_observed, z_ci_lower, z_ci_upper, z_boot.std(),
                                    z_boot.std() / np.sqrt(n_boot), n_boot, mean_infant_age_months,
                                    rho_observed, rho_ci_lower_boot, rho_ci_upper_boot]
master_order = ['infant'] + order
summary_df = summary_df.set_index('age_group').reindex(master_order).reset_index()
summary_df

In [ ]:
summary_df.to_csv('compiled/results/isc_deltaid_developmental_summary.csv', index=False)
np.save('compiled/results/infant_restmovie_isc_deltaid_z_bootstrap_distribution.npy', z_boot)

## Mixed-effects model: z-score ~ log(age) + motion + sex, random intercept per dataset

In [ ]:
def get_par_info(subject):
    row = par_df[par_df['participant_id'] == subject]
    return row.movie_FD.item(), row.rest_FD.item(), row.sex.item()

# NOTE: compiled/info/combined_participant_info.csv is removed from this repo (participant privacy); see REPRODUCIBILITY.md to supply your own copy.
par_df = pd.read_csv('compiled/info/combined_participant_info.csv')
mov_fd = par_df[par_df['dataset'] == 'InfantRestMovie']['movie_FD'].values
rest_fd = par_df[par_df['dataset'] == 'InfantRestMovie']['rest_FD'].values
mean_FD = np.mean([mov_fd, rest_fd])
counts = par_df[par_df['dataset'] == 'InfantRestMovie']['sex'].value_counts()
prop = counts['F'] / counts.sum()

# SEED already set globally above -- np.random.choice calls below are reproducible given that.
bootstrap_df = pd.DataFrame({
    'age_months': np.random.choice(inf_ages, size=len(z_boot)),
    'zscore': z_boot,
    'source': np.repeat('bootstrap', len(z_boot)),
    'dataset': np.repeat('infant_restmovie', len(z_boot)),
    'subject_id': np.repeat('bootstrap', len(z_boot)),
    'mean_FD': np.repeat(mean_FD, len(z_boot)),
    'movie_FD': np.random.choice(mov_fd, size=len(z_boot)),
    'rest_FD': np.random.choice(rest_fd, size=len(z_boot)),
    'sex': np.random.choice(['M', 'F'], size=len(z_boot), p=[1 - prop, prop]),
})

subjects_df = res[['age_months', 'zscore', 'subject_id', 'dataset']].copy()
subjects_df['source'] = 'subject'
info = np.array([get_par_info(s) for s in subjects_df['subject_id'].values])
subjects_df['movie_FD'] = info[:, 0].astype(float)
subjects_df['rest_FD'] = info[:, 1].astype(float)
subjects_df['sex'] = info[:, 2]
subjects_df['mean_FD'] = (subjects_df['movie_FD'] + subjects_df['rest_FD']) / 2
subjects_df['log_age'] = np.log(subjects_df['age_months'] + 1)
summary_df['log_age'] = np.log(summary_df['mean_age_months'] + 1)
combined_df = pd.concat([subjects_df, bootstrap_df], ignore_index=True)
combined_df['log_age'] = np.log(combined_df['age_months'] + 1)
combined_df.head()

In [ ]:
# Mixed effects model fit on real (non-bootstrap) subjects only, random intercept per dataset
model_log = smf.mixedlm('zscore ~ log_age + movie_FD + rest_FD + sex', data=subjects_df,
                         groups=subjects_df['dataset']).fit(reml=False)
print(f"MixedLM AIC: {model_log.aic:.2f}, BIC: {model_log.bic:.2f}")
coef_log_age = model_log.params['log_age']
pval_log_age = model_log.pvalues['log_age']
print(f"Coefficient for log_age: {coef_log_age:.4f}, p-value: {pval_log_age:.3e}")
model_log.summary()

## Final plot: deltaID-ISC relationship across development (with bootstrap CI band)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
mean_col, ci_lower_col, ci_upper_col, point_label = 'mean_z', 'ci_lower', 'ci_upper', 'zscore'

sns.scatterplot(x='age_months', y=point_label, data=combined_df[combined_df['subject_id'] != 'bootstrap'], ax=ax,
                hue='dataset', palette=[helper.dataset_colors('adult_restmovie'), helper.dataset_colors('hbn')],
                s=30, alpha=0.8, zorder=-1)
sns.scatterplot(x='age_months', y=point_label, data=combined_df[combined_df['subject_id'] == 'bootstrap'],
                ax=ax, color=helper.dataset_colors('infant_restmovie'), s=10, zorder=1, alpha=0.4, marker='^',
                label='Infant bootstrap')
plt.legend(title='dataset', labels=['adult_restmovie', 'hbn', 'infant_restmovie'], loc='lower left')

for idx, row in summary_df.iterrows():
    if idx != 0:
        continue
    ax.errorbar(x=row['mean_age_months'], y=row[mean_col],
                yerr=[[row[mean_col] - row[ci_lower_col]], [row[ci_upper_col] - row[mean_col]]],
                fmt='o', c='k', capsize=2, linewidth=2, zorder=12)

infant_age = summary_df.loc[summary_df['age_group'] == 'infant', 'mean_age_months'].values[0]
infant_pred_z = model_log.predict(pd.DataFrame({'log_age': [np.log(infant_age) + 1], 'movie_FD': [mov_fd.mean()],
                                                 'rest_FD': [rest_fd.mean()], 'sex': ['F']})).values[0]
ax.errorbar(x=infant_age + 4, y=infant_pred_z, fmt='o', c='magenta', capsize=4, linewidth=2, zorder=15,
            label='Model prediction at mean infant age')

age_range = np.linspace(res['age_months'].min(), res['age_months'].max(), 100)
log_age_range = np.log(age_range + 1)
predicted_z = model_log.params['Intercept'] + model_log.params['log_age'] * log_age_range
ax.plot(age_range, predicted_z, color='gray')
ax.legend(title='Dataset', loc='lower left')

# --- bootstrap CI band for the regression line (re-enabled; deterministic via random_state=i) ---
n_boot = 1000
boot_preds = np.zeros((n_boot, len(age_range)))
print("Running bootstrap...")
for i in range(n_boot):
    if i % 100 == 0:
        print(f"  {i}/{n_boot}")
    boot_dfs = []
    for ds in subjects_df['dataset'].unique():
        ds_data = subjects_df[subjects_df['dataset'] == ds]
        boot_dfs.append(resample(ds_data, replace=True, random_state=i))
    df_boot = pd.concat(boot_dfs).reset_index(drop=True)
    df_boot['log_age'] = np.log(df_boot['age_months'] + 1)
    try:
        m_boot = smf.mixedlm('zscore ~ log_age + movie_FD + rest_FD + sex', data=df_boot,
                              groups=df_boot['dataset']).fit(reml=False, disp=False)
        boot_preds[i] = (m_boot.fe_params['Intercept'] + m_boot.fe_params['log_age'] * log_age_range)
    except Exception as e:
        print(f"Boot {i} failed: {e}")
        boot_preds[i] = np.nan
print("Bootstrap complete.")

ci_lower = np.nanpercentile(boot_preds, 2.5, axis=0)
ci_upper = np.nanpercentile(boot_preds, 97.5, axis=0)
y_pred = (model_log.fe_params['Intercept'] + model_log.fe_params['log_age'] * log_age_range)

boot_mean = np.nanmean(boot_preds, axis=0)
ci_lower_centered = y_pred - (boot_mean - ci_lower)
ci_upper_centered = y_pred + (ci_upper - boot_mean)

ax.plot(age_range, y_pred, color='gray', linewidth=1.5, zorder=3)
ax.fill_between(age_range, ci_lower_centered, ci_upper_centered, color='gray', alpha=0.2, zorder=2)

ax.text(0.95, 0.25, f'\u03b2={coef_log_age:.3f}***', transform=ax.transAxes, ha='right', va='top', fontsize=10)
ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Age (months)', fontsize=12)
ax.set_ylabel('z-score', fontsize=12)
ax.set_title("deltaID-ISC relationship across development", fontsize=12)
sns.despine()
plt.savefig(f'{PLOT_DIR}/isc_deltaid_developmental_scatter_with_bootstrap_log.pdf', format='pdf',
            transparent=True, bbox_inches='tight')
plt.show()